In [13]:
import psycopg2
from dotenv import load_dotenv
import os
from pathlib import Path
import pandas as pd

In [2]:
import sys

print(sys.executable)

C:\Projects\fsb-projects\mp7_pipeline_toko_bandung\.venv\Scripts\python.exe


In [10]:
BASE_DIR = Path(os.getcwd()).resolve()

load_dotenv(BASE_DIR / ".env")

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database=os.getenv("POSTGRES_DB"),
    user=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD")
)

print("Berhasil terhubung ke PostgreSQL!")

Berhasil terhubung ke PostgreSQL!


In [14]:
import pandas as pd

query = """
SELECT *
FROM orders
LIMIT 10;
"""

df = pd.read_sql_query(query, conn)

df.head()

C:\Users\Alvin Faisal\AppData\Local\Temp\ipykernel_10796\1855603530.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,order_id,customer_id,product_id,qty,unit_price,status,order_ts,updated_at
0,912,C0378,P005,4,50000,paid,2026-10-20 00:08:00,2026-10-20 00:10:00
1,1924,C0347,P030,1,120000,shipped,2026-10-20 00:02:00,2026-10-20 00:31:00
2,540,C0029,P035,4,250000,paid,2026-10-20 00:17:00,2026-10-20 00:45:00
3,1768,C0239,P005,2,120000,paid,2026-10-20 00:45:00,2026-10-20 00:58:00
4,1482,C0258,P025,3,120000,paid,2026-10-20 00:48:00,2026-10-20 01:50:00


In [15]:
query = """
SELECT
    COUNT(*) AS total_rows,
    MIN(updated_at) AS earliest_update,
    MAX(updated_at) AS latest_update
FROM orders;
"""

source_stats = pd.read_sql_query(query, conn)

source_stats

C:\Users\Alvin Faisal\AppData\Local\Temp\ipykernel_10796\1785853786.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  source_stats = pd.read_sql_query(query, conn)


,total_rows,earliest_update,latest_update
0,2000,2026-10-20 00:10:00,2026-10-27 03:28:00


In [16]:
print("Total rows:", source_stats.loc[0, "total_rows"])
print("Earliest update:", source_stats.loc[0, "earliest_update"])
print("Latest update:", source_stats.loc[0, "latest_update"])

Total rows: 2000
Earliest update: 2026-10-20 00:10:00
Latest update: 2026-10-27 03:28:00


In [17]:
query = """
SELECT *
FROM orders
ORDER BY updated_at, order_id;
"""

df = pd.read_sql_query(query, conn)

print("Total data berhasil di-extract:", len(df))
print("Jumlah kolom:", len(df.columns))

df.head()

Total data berhasil di-extract: 2000
Jumlah kolom: 8


C:\Users\Alvin Faisal\AppData\Local\Temp\ipykernel_10796\2669839331.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,order_id,customer_id,product_id,qty,unit_price,status,order_ts,updated_at
0,912,C0378,P005,4,50000,paid,2026-10-20 00:08:00,2026-10-20 00:10:00
1,1924,C0347,P030,1,120000,shipped,2026-10-20 00:02:00,2026-10-20 00:31:00
2,540,C0029,P035,4,250000,paid,2026-10-20 00:17:00,2026-10-20 00:45:00
3,1768,C0239,P005,2,120000,paid,2026-10-20 00:45:00,2026-10-20 00:58:00
4,1482,C0258,P025,3,120000,paid,2026-10-20 00:48:00,2026-10-20 01:50:00


In [18]:
batch_max_updated_at = df["updated_at"].max()

print("Jumlah data batch:", len(df))
print("Watermark kandidat:", batch_max_updated_at)

Jumlah data batch: 2000
Watermark kandidat: 2026-10-27 03:28:00


In [19]:
import hashlib
import json
from datetime import datetime, timezone

# Ubah DataFrame menjadi CSV dalam memory
csv_data = df.to_csv(index=False).encode("utf-8")

# Hitung SHA-256
raw_checksum = hashlib.sha256(csv_data).hexdigest()

# Tentukan lokasi object di MinIO
raw_key = "orders/initial_load/orders.csv"

# Upload ke MinIO
s3.put_object(
    Bucket="toko-raw",
    Key=raw_key,
    Body=csv_data,
    ContentType="text/csv"
)

print("Raw data berhasil ditulis ke MinIO!")
print("Object key:", raw_key)
print("Jumlah baris:", len(df))
print("Ukuran file:", len(csv_data), "bytes")
print("SHA-256:", raw_checksum)

NameError: name 's3' is not defined